cd contest2
source venv/bin/activate
cd ../
export LD_LIBRARY_PATH=$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cudnn/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cublas/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusolver/lib:$VIRTUAL_ENV/lib/python3.10/site-packages/nvidia/cusparse/lib
python3 -c "import tensorflow as tf; print('Доступные GPU:', tf.config.list_physical_devices('GPU'))"
jupyter notebook --no-browser

In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path('.')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
LABELS_PATH = DATA_DIR / 'labels.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

IMG_SIZE = 288
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

I0000 00:00:1779900697.384734    5441 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("включено")
    except RuntimeError as e:
        print(e)

включено


In [4]:
labels = pd.read_csv(LABELS_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

labels['filename'] = labels['id'].astype(str) + '.jpg'
labels['filepath'] = labels['filename'].apply(lambda name: str(TRAIN_DIR / name))

classes = sorted(labels['breed'].unique())
class_to_index = {breed: idx for idx, breed in enumerate(classes)}
labels['label'] = labels['breed'].map(class_to_index)

train_df, valid_df = train_test_split(
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels['breed'],
)


In [5]:
def decode_image(path, label=None):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    if label is None:
        return image
    return image, tf.one_hot(label, depth=len(classes))

def make_dataset(df, training=False):
    paths = df['filepath'].values
    labels_array = df['label'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_array))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df, training=False)

I0000 00:00:1779900727.443977    5441 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.15, seed=SEED),
        layers.RandomTranslation(0.1, 0.1, seed=SEED),
        layers.RandomContrast(factor=0.1, seed=SEED),
    ])



In [7]:
def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)

    base_model = EfficientNetB2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = False 

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x) 
    
    outputs = layers.Dense(
        len(classes), 
        activation='softmax',
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
)


In [8]:
callbacks_phase1 = [
    keras.callbacks.ModelCheckpoint(
        'efficientnetb2_dog_breeds_phase1.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

initial_epochs = 6
history_frozen = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=initial_epochs,
    callbacks=callbacks_phase1,
)



Epoch 1/6


I0000 00:00:1779900746.507369    5517 cuda_dnn.cc:461] Loaded cuDNN version 92200


512/512 ━━━━━━━━━━━━━━━━━━━━ 78s 122ms/step - accuracy: 0.6236 - loss: 1.9526 - top_5_accuracy: 0.8508 - val_accuracy: 0.8910 - val_loss: 0.5758 - val_top_5_accuracy: 0.9941
Epoch 2/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 60s 116ms/step - accuracy: 0.8103 - loss: 0.7859 - top_5_accuracy: 0.9705 - val_accuracy: 0.8998 - val_loss: 0.4432 - val_top_5_accuracy: 0.9936
Epoch 3/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 57s 110ms/step - accuracy: 0.8393 - loss: 0.6553 - top_5_accuracy: 0.9798 - val_accuracy: 0.9051 - val_loss: 0.4195 - val_top_5_accuracy: 0.9946
Epoch 4/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 59s 116ms/step - accuracy: 0.8525 - loss: 0.6004 - top_5_accuracy: 0.9846 - val_accuracy: 0.9042 - val_loss: 0.4200 - val_top_5_accuracy: 0.9927
Epoch 5/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 59s 115ms/step - accuracy: 0.8705 - loss: 0.5476 - top_5_accuracy: 0.9889 - val_accuracy: 0.8963 - val_loss: 0.4283 - val_top_5_accuracy: 0.9936
Epoch 6/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 57s 112ms/step - accuracy: 0.8808 - loss: 0.5204 - 

In [9]:
model.load_weights('efficientnetb2_dog_breeds_phase1.keras')

base_model.trainable = True

fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
)

callbacks_phase2 = [
    keras.callbacks.ModelCheckpoint(
        'efficientnetb2_dog_breeds_best.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

fine_tune_epochs = 15
total_epochs = len(history_frozen.history['loss']) + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=total_epochs,
    initial_epoch=len(history_frozen.epoch),
    callbacks=callbacks_phase2,
)

Epoch 7/21


E0000 00:00:1779901128.396476    5441 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/efficientnetb2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


512/512 ━━━━━━━━━━━━━━━━━━━━ 95s 153ms/step - accuracy: 0.7273 - loss: 1.2255 - top_5_accuracy: 0.9445 - val_accuracy: 0.8758 - val_loss: 0.5535 - val_top_5_accuracy: 0.9927 - learning_rate: 5.0000e-06
Epoch 8/21
512/512 ━━━━━━━━━━━━━━━━━━━━ 78s 152ms/step - accuracy: 0.7601 - loss: 1.0685 - top_5_accuracy: 0.9545 - val_accuracy: 0.8787 - val_loss: 0.5225 - val_top_5_accuracy: 0.9927 - learning_rate: 5.0000e-06
Epoch 9/21
512/512 ━━━━━━━━━━━━━━━━━━━━ 76s 148ms/step - accuracy: 0.7843 - loss: 0.9538 - top_5_accuracy: 0.9662 - val_accuracy: 0.8797 - val_loss: 0.5031 - val_top_5_accuracy: 0.9927 - learning_rate: 5.0000e-06
Epoch 10/21
512/512 ━━━━━━━━━━━━━━━━━━━━ 76s 149ms/step - accuracy: 0.8025 - loss: 0.8776 - top_5_accuracy: 0.9726 - val_accuracy: 0.8841 - val_loss: 0.4857 - val_top_5_accuracy: 0.9932 - learning_rate: 5.0000e-06
Epoch 11/21
512/512 ━━━━━━━━━━━━━━━━━━━━ 76s 149ms/step - accuracy: 0.8095 - loss: 0.8293 - top_5_accuracy: 0.9751 - val_accuracy: 0.8875 - val_loss: 0.4771 -

In [10]:
test_paths = [str(TEST_DIR / f'{image_id}.jpg') for image_id in sample_submission['id']]
test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(lambda path: decode_image(path), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
print('Начинаю обучение модели депу, додепу и мемам')
predictions = model.predict(test_ds, verbose=0)
predictions = np.clip(predictions, 0.0001, 0.9999)
predictions = predictions / np.sum(predictions, axis=1, keepdims=True)

submission = pd.DataFrame(predictions, columns=classes)
submission.insert(0, 'id', sample_submission['id'].values)
submission = submission[sample_submission.columns]
submission.to_csv('submission3v3.csv', index=False)
print('План-скам готов. Ха-ха, заскамил мамонта')



Начинаю обучение модели депу, додепу и мемам
План-скам готов. Ха-ха, заскамил мамонта
